# maritime-co2 quickstart: OPS-avoidable CO2 (liquid bulk)

Estimates the at-berth CO2 that Onshore Power Supply would avoid for tanker calls.

**Auxiliary engine only; boilers excluded** (they keep running for cargo heating).
Power output at berth from IMO Fourth GHG Study 2020, Table 17; SFOC from Table 19.
Sample: 100 real Port of Itaqui liquid-bulk calls, 2024 (~10% of annual calls).

In [ ]:
import pandas as pd
from maritime_co2 import auxiliary_emissions, aux_power_at_berth, estimate_ops_savings, suggest_ship_type

## 1. Single call

In [ ]:
power = aux_power_at_berth(105_000, 'oil_tanker')  # Aframax: 800 kW (Table 17)
r = auxiliary_emissions(power_kw=power, load_factor=1.0, hours=36, fuel='MGO')
print(f'Auxiliary CO2 avoided by OPS: {r.co2_tonnes:.2f} t')

## 2. Real 2024 sample

In [ ]:
calls = pd.read_csv('../data/sample/itaqui_liquid_bulk_2024_sample.csv')
res = estimate_ops_savings(calls)
res[['call_id','dwt','size_band','berth_hours','co2_avoided_tonnes']].head(10)

## 3. Totals and distribution

In [ ]:
print(f"Calls: {len(res)} | Total OPS-avoidable CO2: {res['co2_avoided_tonnes'].sum():.1f} t")
res.groupby('size_band')['co2_avoided_tonnes'].sum().plot(kind='bar', title='OPS-avoidable CO2 by size band (t)')

## 4. By ship type, net of grid emissions

In [ ]:
calls['ship_type'] = calls['product'].map(suggest_ship_type)  # a first guess from the cargo
net = estimate_ops_savings(calls, ship_type_col='ship_type', grid_factor=0.0545)  # t CO2/MWh, Brazil 2024 (MCTI)
net[['co2_avoided_tonnes', 'grid_co2_tonnes', 'net_avoided_tonnes']].sum().round(1)